# 01 — Tiled Matrix Multiplication

**Goal:** calculate `C = A @ B` with a Triton kernel and compare it with PyTorch.

This introduces two-dimensional output tiles and data reuse. It is a more useful deep-learning workload than vector addition because linear layers and many model operations rely on matrix multiplication.

## 1. Environment

Run in Colab with a CUDA GPU. The kernel uses FP16 inputs and FP32 accumulation for the running sum. This is a learning implementation, not a tuned replacement for `torch.matmul`.

In [ ]:
import math
import statistics
import time
import torch
import triton
import triton.language as tl

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Google Colab, then run this notebook again.")

print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 2. The calculation

If `A` has shape `(M, K)` and `B` has shape `(K, N)`, the output has shape `(M, N)`. Each output value is a sum over the shared dimension `K`:

`C[m, n] = sum(A[m, k] * B[k, n] for k in range(K))`

The kernel calculates one rectangular output tile per Triton program. It walks through `K` in smaller chunks called `BLOCK_K` and adds each partial result to an accumulator.

In [ ]:
@triton.jit
def matmul_kernel(A_ptr, B_ptr, C_ptr,
                  M: tl.constexpr, N: tl.constexpr, K: tl.constexpr,
                  BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    k_offsets = tl.arange(0, BLOCK_K)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k_start in range(0, K, BLOCK_K):
        ks = k_start + k_offsets
        a_ptrs = A_ptr + rows[:, None] * K + ks[None, :]
        b_ptrs = B_ptr + ks[:, None] * N + cols[None, :]
        a_mask = (rows[:, None] < M) & (ks[None, :] < K)
        b_mask = (ks[:, None] < K) & (cols[None, :] < N)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)
        acc += tl.dot(a, b)

    c_ptrs = C_ptr + rows[:, None] * N + cols[None, :]
    c_mask = (rows[:, None] < M) & (cols[None, :] < N)
    tl.store(c_ptrs, acc, mask=c_mask)


def triton_matmul(a, b, out, block_m=32, block_n=64, block_k=32):
    m, k = a.shape
    k_b, n = b.shape
    if k != k_b:
        raise ValueError("The inner dimensions of A and B must match")
    grid = (triton.cdiv(m, block_m), triton.cdiv(n, block_n))
    matmul_kernel[grid](a, b, out, m, n, k,
                        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k)
    return out

## 3. Correctness, including partial tiles

The dimensions below are deliberately not all multiples of the tile dimensions. This checks that the load/store masks handle the edges safely.

In [ ]:
M, K, N = 127, 95, 141
a = torch.randn((M, K), device="cuda", dtype=torch.float16)
b = torch.randn((K, N), device="cuda", dtype=torch.float16)
out = torch.empty((M, N), device="cuda", dtype=torch.float16)

triton_matmul(a, b, out)
expected = torch.matmul(a, b)
torch.testing.assert_close(out, expected, atol=3e-2, rtol=3e-2)
print("Correctness check passed:", tuple(out.shape))

## 4. Benchmark one representative size

The first goal is to obtain a trustworthy comparison, not to beat PyTorch. PyTorch's matrix multiplication is already highly optimized. This timing uses preallocated output buffers so the benchmark focuses mainly on the operation.

In [ ]:
def median_gpu_time_ms(fn, warmup=10, repeats=50, rounds=5):
    """Measure repeated calls with CUDA events; return median ms per call and all rounds."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    samples = []
    for _ in range(rounds):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(repeats):
            fn()
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end) / repeats)

    return statistics.median(samples), samples

In [ ]:
M = K = N = 512
a = torch.randn((M, K), device="cuda", dtype=torch.float16)
b = torch.randn((K, N), device="cuda", dtype=torch.float16)
out_torch = torch.empty((M, N), device="cuda", dtype=torch.float16)
out_triton = torch.empty_like(out_torch)


def run_torch():
    torch.matmul(a, b, out=out_torch)


def run_triton():
    triton_matmul(a, b, out_triton, block_m=32, block_n=64, block_k=32)

run_torch()
run_triton()
torch.testing.assert_close(out_triton, out_torch, atol=3e-2, rtol=3e-2)

torch_ms, _ = median_gpu_time_ms(run_torch, repeats=20)
triton_ms, _ = median_gpu_time_ms(run_triton, repeats=20)
flops = 2 * M * N * K
print(f"PyTorch: {torch_ms:.6f} ms/call")
print(f"Triton:  {triton_ms:.6f} ms/call")
print(f"PyTorch / Triton ratio: {torch_ms / triton_ms:.3f}x")
print(f"Triton estimated throughput: {flops / (triton_ms / 1000) / 1e12:.3f} TFLOP/s")

## 5. What to understand

- `BLOCK_M` and `BLOCK_N` set the output tile's height and width.
- `BLOCK_K` sets the amount of the shared dimension processed each loop.
- `acc` stores partial sums in FP32.
- `tl.dot(a, b)` performs the tile matrix multiply.
- A tile can reuse loaded values to calculate several output values, reducing repeated memory loads.

Do not tune these values yet. Notebook 02 focuses on how to design a small, controlled tuning experiment.